# Stage 04 — Data Acquisition and Ingestion

Stage 04. **I based it on the lecture notebook.**

In [ ]:
# Install missing packages (uncomment and run to install).
# !pip install pandas requests yfinance python-dotenv beautifulsoup4

In [ ]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT / ".env.example").exists() and (ROOT.parent / ".env.example").exists():
    ROOT = ROOT.parent

CHECKS = [
    (".env", "NEEDED", "copy .env.example to .env"),
    (".env.example", "NEEDED", "template for local secrets"),
    ("src/io_utils.py", "NEEDED", "save and validate helpers"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

In [ ]:
import sys

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import get_key, load_env
from src.io_utils import save_csv, validate

# Raw outputs.
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Load `.env`.
load_env()
print("ALPHAVANTAGE_API_KEY loaded?", bool(get_key("ALPHAVANTAGE_API_KEY")))

## API pull

In [ ]:
TICKER = "AAPL"
alphavantage_key = get_key("ALPHAVANTAGE_API_KEY")
use_alphavantage = bool(alphavantage_key)
print("Using Alpha Vantage:", use_alphavantage)

alphavantage_json = {}
series_keys = []

if use_alphavantage:
    # Alpha Vantage daily close.
    alphavantage_url = "https://www.alphavantage.co/query"
    alphavantage_params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": TICKER,
        "outputsize": "compact",
        "apikey": alphavantage_key,
        "datatype": "json",
    }
    alphavantage_response = requests.get(alphavantage_url, params=alphavantage_params, timeout=30)
    alphavantage_response.raise_for_status()
    alphavantage_json = alphavantage_response.json()
    # Time series key name can change.
    series_keys = [name for name in alphavantage_json.keys() if "Time Series" in name]
    if not series_keys:
        print("Alpha Vantage returned no series:", str(list(alphavantage_json.values())[0])[:150])
        use_alphavantage = False

if use_alphavantage:
    daily_series = alphavantage_json[series_keys[0]]
    prices = pd.DataFrame(daily_series).T.rename_axis("date").reset_index()
    prices = prices[["date", "4. close"]].rename(columns={"4. close": "close"})
    # Dates and floats.
    prices["date"] = pd.to_datetime(prices["date"])
    prices["close"] = pd.to_numeric(prices["close"])
else:
    # yfinance fallback.
    downloaded = yf.download(TICKER, period="6mo", interval="1d", auto_adjust=False)
    prices = pd.DataFrame(downloaded).reset_index()
    prices = prices.rename(columns={"Date": "date", "Close": "close"})[["date", "close"]]

prices = prices.sort_values("date").reset_index(drop=True)
# Required cols, shape, NA count.
api_check = validate(prices, ["date", "close"])
api_check

In [ ]:
# Save API CSV.
api_csv_path = save_csv(
    prices,
    prefix="api",
    raw_dir=RAW,
    source="alphavantage" if use_alphavantage else "yfinance",
    symbol=TICKER,
)

## Scrape a public table

In [ ]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
headers = {"User-Agent": "AFE-Homework/1.0"}

try:
    # Wikipedia DJIA table.
    wiki_response = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    wiki_response.raise_for_status()
    soup = BeautifulSoup(wiki_response.text, "html.parser")
    table = soup.find("table", class_="wikitable")
    if table is None:
        raise RuntimeError("No wikitable found")

    # Header row then data rows.
    rows = []
    for table_row in table.find_all("tr"):
        cells = [cell.get_text(strip=True) for cell in table_row.find_all(["td", "th"])]
        if cells:
            rows.append(cells)
    header, *data = rows
    scraped = pd.DataFrame(data, columns=header)
except Exception as error:
    print("Scrape failed, using inline demo table:", error)
    html = "<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>"
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for table_row in soup.find_all("tr"):
        cells = [cell.get_text(strip=True) for cell in table_row.find_all(["th", "td"])]
        if cells:
            rows.append(cells)
    header, *data = rows
    scraped = pd.DataFrame(data, columns=header)

# Required cols, shape, NA count.
scrape_check = validate(scraped, list(scraped.columns))
scrape_check

In [ ]:
# Save scrape CSV.
scrape_csv_path = save_csv(scraped, prefix="scrape", raw_dir=RAW, site="wikipedia", table="djia")

## Documentation

- For the API source, I use Alpha Vantage `TIME_SERIES_DAILY` (`symbol=AAPL`, `outputsize=compact`) if `ALPHAVANTAGE_API_KEY` is set. Otherwise, I use yfinance `AAPL` 6-month daily close.
-  My table scrape source is [Dow Jones Industrial Average](https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average) `wikitable` (components).
- I validate required columns, shape, total NA count via `validate()`.
- `.env` is gitignored. I keep `.env.example` in the repo.

Assumptions and risks:
- The Alpha Vantage free tier is 25 calls/day and can return HTTP 200 without data.
- yfinance and Wikipedia HTML can change suddenly.
- Wikipedia can change Table selectors (`class=wikitable`) suddenly.